In [1]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
diabetes_rf = joblib.load('../output/diabetes_rf_model.joblib')
heart_lasso = joblib.load('../output/heart_disease_lasso_model.joblib')
stroke_lasso = joblib.load('../output/stroke_model.joblib')
stroke_encoder = joblib.load('../output/stroke_encoder.joblib')


In [3]:
DIABETES_FEATURES = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
HEART_FEATURES = ["Age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

STROKE_CATEGORICAL_FEATURES = ["gender", "ever_married", "work_type", "smoking_status"]
STROKE_NUMERICAL_FEATURES = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

# Residence_type is dropped in the training pipeline, so it is not part of the final attack surface.
STROKE_ENCODED_FEATURES = STROKE_NUMERICAL_FEATURES + list(
    stroke_encoder.get_feature_names_out(STROKE_CATEGORICAL_FEATURES)
)
stroke_coef = stroke_lasso.coef_[0]

def collapse_stroke_feature(feature_name):
    for column in STROKE_CATEGORICAL_FEATURES:
        if feature_name.startswith(f"{column}_"):
            return column
    return feature_name


In [16]:
# Analyze Random Forest model for Diabetes risk prediction
rf_importances = diabetes_rf.feature_importances_
df_rf = pd.DataFrame({'Feature': DIABETES_FEATURES, 'Importance': rf_importances})
df_rf = df_rf.sort_values('Importance', ascending=False)
print("Top 3 Attack Vectors (by Gini Importance):")
print(df_rf.head(3).to_string(index=False))

Top 3 Attack Vectors (by Gini Importance):
Feature  Importance
Glucose    0.264336
    BMI    0.159328
    Age    0.154988


In [17]:
# Analyze Lasso LR model (Heart Disease - C=0.1)
heart_coef = heart_lasso.coef_[0]
df_heart = pd.DataFrame({'Feature': HEART_FEATURES, 'Coefficient': heart_coef, 'Absolute_Weight': np.abs(heart_coef)})
df_heart = df_heart.sort_values('Absolute_Weight', ascending=False)
zeroed_heart = (df_heart['Coefficient'] == 0).sum()
print(f"Total Features: {len(HEART_FEATURES)}")
print(f"Features Zeroed out by Lasso (Invulnerable to attack): {zeroed_heart}")
print("Top 3 Attack Vectors (by absolute weight):")
print(df_heart[df_heart['Coefficient'] != 0].head(3)[['Feature', 'Coefficient']].to_string(index=False))

Total Features: 13
Features Zeroed out by Lasso (Invulnerable to attack): 2
Top 3 Attack Vectors (by absolute weight):
Feature  Coefficient
     cp     0.702687
oldpeak    -0.642355
     ca    -0.620334


In [ ]:
# Analyze Lasso LR (Stroke - C=0.01)
df_stroke = pd.DataFrame({
    "Feature": STROKE_ENCODED_FEATURES,
    "Coefficient": stroke_coef,
    "Absolute_Weight": np.abs(stroke_coef)
})
df_stroke = df_stroke.sort_values("Absolute_Weight", ascending=False)
df_stroke["Feature_Group"] = df_stroke["Feature"].apply(collapse_stroke_feature)

zeroed_stroke = (df_stroke["Coefficient"] == 0).sum()
active_stroke = df_stroke[df_stroke["Coefficient"] != 0]

grouped_stroke = (
    active_stroke.groupby("Feature_Group", as_index=False)["Absolute_Weight"]
    .sum()
    .sort_values("Absolute_Weight", ascending=False)
)

print(f"Total Encoded Features: {len(STROKE_ENCODED_FEATURES)}")
print(f"Features Zeroed out by Lasso (Invulnerable to attack): {zeroed_stroke}")
print(f"Sparsity Ratio: {(zeroed_stroke / len(STROKE_ENCODED_FEATURES)) * 100:.1f}% off the attack surface.")

print("Top 5 Attack Vectors (by absolute weight):")
print(active_stroke.head(5)[["Feature", "Coefficient"]].to_string(index=False))

print("\nTop Raw Feature Groups (aggregated encoded weights):")
print(grouped_stroke.head(5).to_string(index=False))



Stroke (Lasso LR, C=0.01)
Total Encoded Features: 14
Features Zeroed out by Lasso (Invulnerable to attack): 9
Sparsity Ratio: 64.3% off the attack surface.
Top 5 Attack Vectors (by absolute weight):
                    Feature  Coefficient
                        age     1.046369
               hypertension     0.400282
              heart_disease     0.233349
smoking_status_never smoked    -0.083614
          avg_glucose_level     0.080268

Top Raw Feature Groups (aggregated encoded weights):
    Feature_Group  Absolute_Weight
              age         1.046369
     hypertension         0.400282
    heart_disease         0.233349
   smoking_status         0.083614
avg_glucose_level         0.080268


Hypothesis Confirmed. The stricter C=0.01 penalty on the stroke model severely restricts the attacker's surface area compared to the heart model.